<a href="https://colab.research.google.com/github/PSS-Grp/pss-pub/blob/main/attendance_on_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 勤怠管理アプリ（Flask）を Colab で動かす

このノートブックは、GitHubリポジトリ `PSS-Grp/pss-pub` の `怠/attendance` にあった
Flask製の勤怠管理アプリを、**Google Colab上で動作確認できるように修正したもの**を
実行するためのものです。

## 今回の改修で直した主な不具合・懸念点

1. **`datetime.py` が標準ライブラリと名前衝突（致命的バグ）**
   attendanceフォルダ内に `datetime.py` という自作ファイルがあり、
   `import datetime` と書くたびに標準ライブラリではなくこのファイル自身が
   読み込まれてしまい、`AttributeError` でアプリ全体が起動できない状態でした。
   → 使われていない不要なファイルだったため削除しました。

2. **`login.py` にシェルコマンドが直接書かれていて構文エラー**
   `flask db init` などターミナルで打つはずのコマンドがPythonコードとして
   そのまま書かれており、実行・importすると構文エラーになる状態でした。
   → 実行しても安全なメモ用ファイルに直しました（DB作成は元々 `admin.py` の
   `db.create_all()` で行われており、このファイル自体は使われていません）。

3. **SECRET_KEY・パスワードのハードコード**
   `app.secret_key` や `SECRET_KEY` に固定文字列が、`auth.py` には実際の
   パスワードが平文でそのままコードに書かれ、公開リポジトリに残っていました。
   → 環境変数から読み込み、無ければ安全なランダム値を自動生成する方式に変更。
   `auth.py` は環境変数 or 入力プロンプトからパスワードを受け取る方式に変更しました。

4. **DBファイルパスがカレントディレクトリ依存**
   `sqlite:///db/attendance.db` という相対パスで、実行場所によっては
   DBファイルが見つからず失敗する状態でした。
   → スクリプト自身の場所を基準にした絶対パスに変更しました。

5. **新しいバージョンのFlask-SQLAlchemy / Flask-Adminとの非互換**
   最新版のライブラリでは `db.create_all()` にアプリケーションコンテキストが
   必須になっていたり、`Admin(..., template_mode='bootstrap4')` という書き方が
   廃止されていたため、現在のバージョンに合わせて修正しました。

6. **元のSQLiteデータベースに実データが入ったままコミットされていた**
   `db/attendance.db` に実際の従業員番号・氏名・パスワードハッシュ・勤怠記録が
   含まれていた可能性があるため、このノートブックには含めていません。
   代わりに `seed_demo_user.py` が、動作確認用のダミーユーザーを1件だけ作成します。

## 使い方

1. 下のセルを順番に実行してください。
2. 「ファイルを選択」が表示されたら、お渡しした `attendance_fixed.zip` を
   アップロードしてください。
3. 最後のセルを実行すると、ノートブック内にアプリの画面が埋め込まれます。
   - 従業員番号: `0001`
   - パスワード: `demo1234`


## 1. 修正済みファイル（zip）をアップロード

In [9]:
import os

if os.path.isdir("attendance_fixed"):
    print("attendance_fixed フォルダは既にあります。再アップロードは不要です。")
else:
    try:
        from google.colab import files
        print("attendance_fixed.zip を選択してアップロードしてください。")
        uploaded = files.upload()
        zip_name = next(iter(uploaded))
    except ImportError:
        # Colab以外（ローカルJupyter等）で実行する場合は、
        # あらかじめ同じフォルダに attendance_fixed.zip を置いておく
        zip_name = "attendance_fixed.zip"

    import zipfile
    with zipfile.ZipFile(zip_name, "r") as z:
        z.extractall(".")
    print("展開しました:", os.listdir("attendance_fixed"))


attendance_fixed.zip を選択してアップロードしてください。


Saving attendance_fixed.zip to attendance_fixed.zip
展開しました: ['db', 'tsuya.py', 'admin.py', 'auth.py', 'requirements.txt', '__init__.py', 'run_colab.py', 'honso.py', 'templates', 'models.py', 'index.py', 'login.py', 'seed_demo_user.py', 'static']


## 2. 依存ライブラリのインストール

In [10]:
!pip install -q -r attendance_fixed/requirements.txt
print("インストール完了")


インストール完了


## 3. データベースの初期化 & デモユーザー作成

前述の通り、実際の従業員データを含む元のDBは使わず、
このノートブック用に空のDBとデモユーザー（従業員番号 `0001` / パスワード `demo1234`）を
新規作成します。

In [11]:
import sys, os

APP_DIR = os.path.abspath("attendance_fixed")
if APP_DIR not in sys.path:
    sys.path.insert(0, APP_DIR)

# db/attendance.db が残っていれば一度リセットして作り直す場合はコメントを外す
# os.remove(os.path.join(APP_DIR, "db", "attendance.db"))

os.chdir(APP_DIR)
import seed_demo_user
seed_demo_user.main()


デモユーザー（従業員番号: 0001）は既に存在します。


## 4. アプリを起動して画面を表示

Flaskサーバーをバックグラウンドスレッドで起動し、Colab内蔵の機能で
ノートブック上にアプリの画面を埋め込みます（ngrokなどの外部サービスは不要です）。

ログイン画面が表示されたら、従業員番号 `0001` / パスワード `demo1234` でログインできます。

In [12]:
import run_colab
run_colab.start(port=5000)


サーバーは既に起動しています（ポート 5000）。


<IPython.core.display.Javascript object>

## 5. データベース管理画面（Flask-Admin）を開く

`/admin` にアクセスすると、User（従業員）やTime（勤怠記録）のテーブルをブラウザから
直接閲覧・編集できるFlask-Adminの管理画面が開きます。

埋め込みiframeには通常のブラウザのようなアドレスバーが無いため、
「URLの末尾を書き換える」ことができません。代わりに、下のセルを実行すると
`/admin` を指定したiframeを新しく表示します（サーバーは3.で起動済みのものを
そのまま使うので、再起動の必要はありません）。

In [13]:
try:
    from google.colab import output
    output.serve_kernel_port_as_iframe(5000, path="/admin", height=720)
except ImportError:
    print("Colab環境ではないため、ブラウザで http://127.0.0.1:5000/admin を開いてください。")


<IPython.core.display.Javascript object>

補足: どうしても通常のブラウザタブ（アドレスバーあり）で開きたい場合は、
上に表示されたiframe内を右クリックし、「このフレームを新しいタブで開く」
（ブラウザにより表記は異なります）を選ぶと、そのURLが新しいタブで開きます。
そのアドレスバーの末尾を `/admin` などに書き換えれば、他のページにも移動できます。

## 補足

- ログイン用のパスワードハッシュを新しく作りたい場合は、ターミナルの代わりに
  次のセルのようにして `auth.py` の関数を呼び出せます。
- 本番環境として使う場合は、`ATTENDANCE_SECRET_KEY` 環境変数を固定値で設定し、
  実データの取り扱いについて改めてセキュリティ面を確認してください。

In [14]:
import os
os.environ["AUTH_PASSWORD"] = "hujiko0-"
!python auth.py


pw_hash = scrypt:32768:8:1$F8Aj0E9acQKJEUMl$9b0685ee8251818aa484c7ce59af06f9b0ff459b0c0c06c7e718b3f98f82e3ef72bd59b4e9e3e11548896ed87ce0e2149f73b2a79a0dd48999d165dcd84b7cea
